In [4]:
# Held-Out Evaluation of All Substantive Models
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Fixed random seed for reproducibility
SEED = 42
# LOAD DATASET

PATH = r"C:\Users\Sekomane\Desktop\Data ScienceTUT\NIWIS_Water Supply Reliability - Population_26-Sep-2026.csv"

df = pd.read_csv(PATH)

print("=" * 70)
print("DATASET LOADED")
print("=" * 70)
print("Dataset shape:", df.shape)

# CLEAN DATA

data = df.copy()

# Remove empty CSV index column, if present
data = data.loc[:, ~data.columns.str.contains("^Unnamed")]

# Standardise column names
# This fixes the double spaces found in the original NIWIS CSV.
data.columns = (
    data.columns
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
)

def clean_number(series):
    return pd.to_numeric(
        series.astype(str)
              .str.replace(" ", "", regex=False)
              .str.replace(",", "", regex=False)
              .str.strip(),
        errors="coerce"
    )


# Convert count/population columns to numeric
for col in data.columns:

    if "%" not in col and col != "WSA Name":
        data[col] = clean_number(data[col])


# Convert percentage columns
for col in data.columns:

    if "%" in col:

        data[col] = pd.to_numeric(
            data[col]
                .astype(str)
                .str.replace("%", "", regex=False)
                .str.strip(),
            errors="coerce"
        )



# CREATE TARGET

# Binary classification:
#
# Class 0 = Reliable
#           At least 60% reliable supply
#
# Class 1 = Low Reliability
#           Less than 60% reliable supply

data["low_reliability"] = (
    data["% Population Reliable Supply"] < 60
).astype(int)


print("\n" + "=" * 70)
print("TARGET DISTRIBUTION")
print("=" * 70)

print(data["low_reliability"].value_counts().sort_index())

print("\nTarget percentages:")

print(
    data["low_reliability"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(1)
)



# REMOVE TARGET LEAKAGE

# These columns directly contain or reconstruct information about
# reliable water supply and therefore must NOT be predictors.

leakage_columns = [
    "Population Reliable Supply",
    "% Population Reliable Supply",
    "Urban Population Reliable Supply",
    "% Urban Population Reliable Supply",
    "Rural Population Reliable Supply",
    "% Rural Population Reliable Supply"
]


# Target
y = data["low_reliability"]


# Predictors
# Remove:
# - target
# - WSA identifier
# - leakage variables

X = data.drop(
    columns=[
        "low_reliability",
        "WSA Name"
    ] + leakage_columns,
    errors="ignore"
).copy()



# FEATURE ENGINEERING

# Overall population access rate
X["population_access_rate"] = (
    X["Population with access to water"]
    / X["Population"]
) * 100


# Urban water access rate
X["urban_access_rate"] = (
    X["Urban Population with access to water"]
    / X["Urban Population"]
) * 100


# Rural water access rate
X["rural_access_rate"] = (
    X["Rural Population with access to water"]
    / X["Rural Population"]
) * 100


# Urban population share
X["urban_population_share"] = (
    X["Urban Population"]
    / X["Population"]
) * 100


# Rural population share
X["rural_population_share"] = (
    X["Rural Population"]
    / X["Population"]
) * 100


# Division by zero may create infinity.
# Convert these values to NaN so they can be imputed.
X = X.replace(
    [np.inf, -np.inf],
    np.nan
)


print("\n" + "=" * 70)
print("FEATURES USED")
print("=" * 70)

for feature in X.columns:
    print("-", feature)



# DEVELOPMENT / HELD-OUT TEST SPLIT

# 80% development
# 20% held-out test
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)


print("\n" + "=" * 70)
print("DATA SPLIT")
print("=" * 70)

print("Full dataset :", len(X))
print("Development  :", len(X_dev))
print("Test         :", len(X_test))


print("\nDevelopment class distribution:")
print(y_dev.value_counts().sort_index())


print("\nTest class distribution:")
print(y_test.value_counts().sort_index())

# PREPROCESSING
preprocessor = Pipeline([
    
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "scaler",
        RobustScaler()
    )
])



# LOGISTIC REGRESSION

logistic_model = Pipeline([

    (
        "preprocessor",
        preprocessor
    ),

    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=SEED
        )
    )

])


logistic_model.fit(
    X_dev,
    y_dev
)

print("\nLogistic Regression trained successfully.")

#  RANDOM FOREST
rf_model = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "classifier",
        RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        )
    )

])


rf_model.fit(
    X_dev,
    y_dev
)

print("Random Forest trained successfully.")

# MLP

mlp_model = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "scaler",
        RobustScaler()
    ),

    (
        "classifier",
        MLPClassifier(

            hidden_layer_sizes=(32, 16),

            activation="relu",

            solver="adam",

            alpha=0.001,

            batch_size=16,

            learning_rate_init=0.001,

            max_iter=1000,

            early_stopping=True,

            validation_fraction=0.20,

            n_iter_no_change=30,

            random_state=SEED
        )
    )

])


mlp_model.fit(
    X_dev,
    y_dev
)

print(
    "MLP trained successfully. Epochs:",
    mlp_model.named_steps["classifier"].n_iter_
)
# DEEP MLP
deep_mlp_model = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    ),

    (
        "scaler",
        RobustScaler()
    ),

    (
        "classifier",
        MLPClassifier(

            hidden_layer_sizes=(64, 32, 16),

            activation="relu",

            solver="adam",

            alpha=0.005,

            batch_size=16,

            learning_rate_init=0.001,

            max_iter=1200,

            early_stopping=True,

            validation_fraction=0.20,

            n_iter_no_change=30,

            random_state=SEED
        )
    )

])

deep_mlp_model.fit(
    X_dev,
    y_dev
)

print(
    "Deep MLP trained successfully. Epochs:",
    deep_mlp_model.named_steps["classifier"].n_iter_
)

# HELD-OUT MODEL EVALUATION

models = {

    "Logistic Regression":
        logistic_model,

    "Random Forest":
        rf_model,

    "MLP":
        mlp_model,

    "Deep MLP":
        deep_mlp_model
}


results = []

# Save predictions and probabilities.
# These can also be reused in Question 4.2
test_predictions = {}
test_probabilities = {}


for model_name, model in models.items():
    # Predicted class
    y_pred = model.predict(
        X_test
    )

    # Probability of Class 1 = Low Reliability
    y_prob = model.predict_proba(
        X_test
    )[:, 1]


    # Save results for later questions
    test_predictions[model_name] = y_pred
    test_probabilities[model_name] = y_prob

    # Calculate metrics
    accuracy = accuracy_score(
        y_test,
        y_pred
    )


    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )


    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )


    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )


    weighted_f1 = f1_score(
        y_test,
        y_pred,
        average="weighted"
    )


    roc_auc = roc_auc_score(
        y_test,
        y_prob
    )


    pr_auc = average_precision_score(
        y_test,
        y_prob
    )

    # Store results

    results.append({

        "Model":
            model_name,

        "Accuracy":
            accuracy,

        "Precision":
            precision,

        "Recall":
            recall,

        "Macro-F1":
            macro_f1,

        "Weighted-F1":
            weighted_f1,

        "ROC-AUC":
            roc_auc,

        "PR-AUC":
            pr_auc
    })

# 14. CREATE FINAL Q4.1 RESULTS TABLE

results_df = pd.DataFrame(
    results
)


# Columns that need rounding
metric_columns = [

    "Accuracy",
    "Precision",
    "Recall",
    "Macro-F1",
    "Weighted-F1",
    "ROC-AUC",
    "PR-AUC"
]


# Round to 3 decimal places
results_df[metric_columns] = (
    results_df[metric_columns]
    .round(3)
)


print("\n" + "=" * 90)

print(
    "QUESTION 4.1 - HELD-OUT TEST PERFORMANCE"
)

print("=" * 90)


display(
    results_df
)


# 15. DISPLAY BEST MACRO-F1
best_index = (
    results_df["Macro-F1"]
    .idxmax()
)


best_model = (
    results_df.loc[
        best_index,
        "Model"
    ]
)


best_macro_f1 = (
    results_df.loc[
        best_index,
        "Macro-F1"
    ]
)


print(
    "\nHighest held-out Macro-F1:",
    best_model,
    "=",
    best_macro_f1
)


print(
    "\nNOTE: Held-out test performance is reported for evaluation only."
)

print(
    "The test set must not be used for hyperparameter tuning or "
    "provisional champion selection."
)

DATASET LOADED
Dataset shape: (144, 14)

TARGET DISTRIBUTION
low_reliability
0    105
1     39
Name: count, dtype: int64

Target percentages:
low_reliability
0    72.9
1    27.1
Name: proportion, dtype: float64

FEATURES USED
- Population
- Population with access to water
- Urban Population
- Urban Population with access to water
- Rural Population
- Rural Population with access to water
- population_access_rate
- urban_access_rate
- rural_access_rate
- urban_population_share
- rural_population_share

DATA SPLIT
Full dataset : 144
Development  : 115
Test         : 29

Development class distribution:
low_reliability
0    84
1    31
Name: count, dtype: int64

Test class distribution:
low_reliability
0    21
1     8
Name: count, dtype: int64

Logistic Regression trained successfully.
Random Forest trained successfully.
MLP trained successfully. Epochs: 50
Deep MLP trained successfully. Epochs: 35

QUESTION 4.1 - HELD-OUT TEST PERFORMANCE


,Model,Accuracy,Precision,Recall,Macro-F1,Weighted-F1,ROC-AUC,PR-AUC
0,Logistic Regression,0.862,0.833,0.625,0.812,0.855,0.923,0.858
1,Random Forest,0.828,0.714,0.625,0.775,0.824,0.932,0.850
2,MLP,0.862,0.833,0.625,0.812,0.855,0.911,0.853
3,Deep MLP,0.897,0.857,0.750,0.865,0.894,0.875,0.839



Highest held-out Macro-F1: Deep MLP = 0.865

NOTE: Held-out test performance is reported for evaluation only.
The test set must not be used for hyperparameter tuning or provisional champion selection.
